In [1]:
!pip -q install transformers sentence-transformers faiss-cpu sentencepiece torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 68.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [3]:
documents = [
    """Generative Artificial Intelligence is a branch of AI that creates new
    content such as text, images, audio, video and computer programs.""",

    """Large Language Models are transformer-based models trained on massive
    text datasets. They are used for text generation, summarization,
    translation, question answering and conversational AI.""",

    """Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.""",

    """Vector databases store high-dimensional embeddings and perform
    similarity searches. Examples include FAISS, ChromaDB,
    Pinecone, Weaviate and Milvus.""",

    """Prompt engineering is the process of designing clear instructions
    that guide a language model to produce accurate and useful responses.
    Common techniques include zero-shot, few-shot and role-based prompting.""",

    """Fine-tuning adapts a pretrained language model to a specific domain
    or task by training it further using a smaller domain-specific dataset."""
]

In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

document_embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
)

document_embeddings = document_embeddings.astype("float32")

faiss.normalize_L2(document_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
dimension = document_embeddings.shape[1]

vector_database = faiss.IndexFlatIP(dimension)

vector_database.add(document_embeddings)

In [7]:
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCaus

In [8]:
def retrieve_documents(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = vector_database.search(query_embedding, top_k)

    retrieved = []

    for index, score in zip(indices[0], scores[0]):
        retrieved.append({
            "document": documents[index],
            "score": float(score)
        })

    return retrieved

In [9]:
def generate_answer(query, retrieved_documents):

    context = "\n\n".join(
        doc["document"] for doc in retrieved_documents
    )

    prompt = f"""
Answer the question using ONLY the information below.

Context:
{context}

Question:
{query}

If the answer is unavailable, say:
"The answer is not available in the knowledge base."

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )

    return result[0]["generated_text"]

In [11]:
print("="*60)
print("RETRIEVAL AUGMENTED GENERATION (RAG)")
print("="*60)

query = input("Enter your question: ")

retrieved_docs = retrieve_documents(query)

answer = generate_answer(query, retrieved_docs)

print("\nRetrieved Documents")
print("-"*60)

for i, item in enumerate(retrieved_docs, start=1):
    print(f"\nDocument {i}")
    print(item["document"])
    print(f"Similarity Score: {item['score']:.4f}")

print("\nGenerated Answer")
print("-"*60)
print(answer)

RETRIEVAL AUGMENTED GENERATION (RAG)
Enter your question: What is Retrieval-Augmented Generation?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Documents
------------------------------------------------------------

Document 1
Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.
Similarity Score: 0.6933

Document 2
Generative Artificial Intelligence is a branch of AI that creates new
    content such as text, images, audio, video and computer programs.
Similarity Score: 0.3435

Generated Answer
------------------------------------------------------------

Answer the question using ONLY the information below.

Context:
Retrieval-Augmented Generation combines information retrieval with
    text generation. It retrieves relevant documents from an external
    knowledge base and gives them to a language model as context.

Generative Artificial Intelligence is a branch of AI that creates new
    content such as text, images, audio, video and computer programs.

Question:
Wh